# 10 · Ablation Studies (Does Every Part Earn Its Place?)

**Goal.** Remove or swap each component of the model, one at a time, and see what happens.

**Why it matters.** A complex model is only justified if each part helps. Ablation is the honest test:
if removing a piece doesn't hurt, it shouldn't be there.

**What happens below.** Turn parts on/off (routing, gating, fusion, loss), run a small hyper-parameter
search, and record speed/size.

**Output:** `data/stage3_model/ablation_report.json` + efficiency numbers.

## 0. Environment & dependencies

In [1]:
import os, sys, json, time, math, logging, warnings, random
from pathlib import Path
from collections import Counter

for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass

import numpy as np

def _try(name):
    try:
        __import__(name); return True
    except Exception:
        return False

assert _try("torch") and _try("transformers") and _try("sklearn"), "needs torch+transformers+sklearn"
HAS_OPTUNA = _try("optuna")

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("abl")
PART_VERSION = "1.0"
SEED = 42
random.seed(SEED); np.random.seed(SEED)
log.info("optuna available = %s", HAS_OPTUNA)

hinglish-mental-health-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-08-08 07:54:15,314 | INFO | optuna available = False


## 1. Configuration + load frozen features (reuse notebook 09 inputs; emotion embeddings cached)

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(SEED)

def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage3_model").exists() or (d / "data" / "stage2_semantic").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
SEM_DIR  = PROJECT_ROOT / "data" / "stage2_semantic"
TEMP_DIR = PROJECT_ROOT / "data" / "stage1_temporal"
OUTPUT_DIR = PROJECT_ROOT / "data" / "stage3_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}
N_CLASSES = 4
TEMP_FEATS = ["month_t", "day_t", "hour_t", "weekday_t", "is_working_hour_t", "is_weekend_t"]

# epochs reduced vs 3.2's 60 to bound the many-run battery (deltas still indicative)
ABL_EPOCHS = 40
OPT_EPOCHS = 30
N_TRIALS = 15
BATCH = 16

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(SEM_DIR / "corpus_preprocessed_train.jsonl")
test_rows  = load_jsonl(SEM_DIR / "corpus_preprocessed_test.jsonl")
assert set(r["platform"] for r in train_rows) <= {"youtube", "synthetic"}, "TRAIN leak!"
assert set(r["platform"] for r in test_rows)  <= {"reddit", "twitter"},   "TEST leak!"

def muril_matrix(split, rows):
    emb = np.load(SEM_DIR / f"embeddings_{split}.npy").astype(np.float32)
    idx = json.loads((SEM_DIR / f"embeddings_index_{split}.json").read_text(encoding="utf-8"))
    id2row = {rid: i for i, rid in enumerate(idx)}
    return np.vstack([emb[id2row[r["record_id"]]] for r in rows])

def temporal_matrix(rows, split):
    by_id = {r["record_id"]: r for r in load_jsonl(TEMP_DIR / f"temporal_features_{split}.jsonl")}
    feats = np.array([[float(by_id.get(r["record_id"], {}).get(k, 0.0)) for k in TEMP_FEATS] for r in rows], np.float32)
    mask = np.array([float(by_id.get(r["record_id"], {}).get("temporal_mask", r.get("temporal_mask", 1))) for r in rows], np.float32)
    return feats, mask

MURIL_TR = muril_matrix("train", train_rows); MURIL_TE = muril_matrix("test", test_rows)
EMO_TR = np.load(OUTPUT_DIR / "emo_embeddings_train.npy"); EMO_TE = np.load(OUTPUT_DIR / "emo_embeddings_test.npy")
EMO_MODEL_ID = json.loads((OUTPUT_DIR / "emo_meta.json").read_text(encoding="utf-8"))["model_id"]
TEMP_TR, MASK_TR = temporal_matrix(train_rows, "train"); TEMP_TE, MASK_TE = temporal_matrix(test_rows, "test")
y_train = np.array([int(r["tier"]) for r in train_rows]); y_test = np.array([int(r["tier"]) for r in test_rows])
TEXT_DIM, EMO_DIM = MURIL_TR.shape[1], EMO_TR.shape[1]
log.info("loaded: train=%d test=%d | MuRIL=%d emo=%d (%s) | reuse cached emo embeddings",
         len(train_rows), len(test_rows), TEXT_DIM, EMO_DIM, EMO_MODEL_ID)

2026-08-08 07:54:15,705 | INFO | loaded: train=1971 test=2768 | MuRIL=768 emo=768 (MilaNLProc/xlm-emo-t) | reuse cached emo embeddings


## 2. The notebook 09 architecture (verbatim — the system under test) + losses

In [3]:
def squash(s, dim=-1, eps=1e-8):
    n2 = (s * s).sum(dim=dim, keepdim=True)
    return (n2 / (1.0 + n2)) * (s / torch.sqrt(n2 + eps))

class CapsuleRouting(nn.Module):
    def __init__(self, n_in, d_in, n_out, d_out, routing="static", iters=3):
        super().__init__()
        self.n_in, self.n_out, self.routing, self.iters = n_in, n_out, routing, iters
        self.W = nn.Parameter(0.01 * torch.randn(1, n_in, n_out, d_out, d_in))
        self.b_static = nn.Parameter(torch.zeros(1, n_in, n_out))
    def forward(self, u):
        B = u.size(0)
        u_hat = torch.matmul(self.W, u.unsqueeze(2).unsqueeze(-1)).squeeze(-1)
        if self.routing == "static":
            c = torch.softmax(self.b_static, dim=2).unsqueeze(-1)
            return squash((c * u_hat).sum(dim=1))
        b = torch.zeros(B, self.n_in, self.n_out, device=u.device)
        for it in range(self.iters):
            c = torch.softmax(b, dim=2).unsqueeze(-1)
            v = squash((c * u_hat).sum(dim=1))
            if it < self.iters - 1:
                b = b + (u_hat * v.unsqueeze(1)).sum(-1)
        return v

class DriftCapsNet(nn.Module):
    def __init__(self, text_dim, emo_dim, prim_caps=32, prim_dim=16, class_dim=16, d=128,
                 gate="elu", fusion="attention", routing="static", dropout=0.3, temp_dim=6):
        super().__init__()
        self.gate_kind, self.fusion, self.d = gate, fusion, d
        self.prim_caps, self.prim_dim = prim_caps, prim_dim
        self.text_proj = nn.Linear(text_dim, prim_caps * prim_dim)
        self.gate_lin  = nn.Linear(prim_caps * prim_dim, prim_caps * prim_dim)
        self.caps = CapsuleRouting(prim_caps, prim_dim, N_CLASSES, class_dim, routing=routing)
        self.q_proj = nn.Linear(N_CLASSES * class_dim, d)
        self.e_proj = nn.Linear(emo_dim, d)
        self.t_proj = nn.Sequential(nn.Linear(temp_dim, 64), nn.ReLU(), nn.Linear(64, d))
        self.drop = nn.Dropout(dropout); self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d if fusion in ("attention", "text_only") else 3 * d, N_CLASSES)
    def gate(self, H):
        if self.gate_kind == "none":    return H
        if self.gate_kind == "sigmoid": return H * torch.sigmoid(self.gate_lin(H))
        return H * (F.elu(self.gate_lin(H)) + 1.0)
    def temporal_rep(self, x_t, mask):
        return self.t_proj(x_t) * mask.unsqueeze(-1)
    def forward(self, x_text, x_emo, x_temp, mask, return_z=False):
        H = self.drop(F.elu(self.text_proj(x_text)))
        H_g = self.gate(H)
        U = squash(H_g.view(-1, self.prim_caps, self.prim_dim))
        V = self.caps(U).reshape(x_text.size(0), -1)
        q = self.q_proj(V); e = self.e_proj(x_emo); tau = self.temporal_rep(x_temp, mask)
        if self.fusion == "attention":
            kv = torch.stack([tau, e], dim=1)
            att = torch.softmax(torch.matmul(q.unsqueeze(1), kv.transpose(1, 2)) / math.sqrt(self.d), dim=-1)
            z = self.norm(q + torch.matmul(att, kv).squeeze(1))
        elif self.fusion == "concat":
            z = torch.cat([q, tau, e], dim=-1)
        else:
            z = q
        logits = self.head(z)
        return (logits, F.normalize(z, dim=-1)) if return_z else logits

class FocalLoss(nn.Module):
    def __init__(self, weight, gamma=2.0):
        super().__init__(); self.weight = weight; self.gamma = gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits, dim=-1); p = logp.exp(); w = self.weight[y]
        pt = p.gather(1, y.unsqueeze(1)).squeeze(1).clamp(1e-6, 1.0)
        logpt = logp.gather(1, y.unsqueeze(1)).squeeze(1)
        return (-w * (1 - pt) ** self.gamma * logpt).mean()

def supervised_contrastive(z, y, temp=0.1):
    sim = torch.matmul(z, z.t()) / temp; B = z.size(0)
    m = ~torch.eye(B, dtype=torch.bool, device=z.device)
    pos = (y.unsqueeze(0) == y.unsqueeze(1)) & m
    sim = sim.masked_fill(~m, -1e9)
    logp = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    pc = pos.sum(1); valid = pc > 0
    if valid.sum() == 0:
        return torch.tensor(0.0, device=z.device)
    return -((logp.masked_fill(~pos, 0.0).sum(1)[valid]) / pc[valid].clamp(min=1)).mean()

log.info("architecture + losses ready")

2026-08-08 07:54:15,733 | INFO | architecture + losses ready


## 3. Train/eval driver — one config end-to-end (frozen backbones, head only)

In [4]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (accuracy_score, f1_score, recall_score, matthews_corrcoef, roc_auc_score)

DEF = dict(prim_caps=32, prim_dim=16, class_dim=16, fusion_dim=128, routing="static", gate="elu",
           fusion="attention", loss="focal_contrastive", lr=1e-3, dropout=0.3, contrast_w=0.2,
           weight_decay=1e-4, focal_gamma=2.0)

def make_loaders(Xt, Xe, Xtau, M, Y, batch=BATCH, shuffle=True):
    ds = TensorDataset(torch.tensor(Xt), torch.tensor(Xe), torch.tensor(Xtau),
                       torch.tensor(M), torch.tensor(Y).long())
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, generator=g)

def class_weights(y):
    c = np.bincount(y, minlength=4).astype(float)
    return torch.tensor(c.sum() / (4.0 * np.clip(c, 1, None)), dtype=torch.float, device=DEVICE)

def train_eval(cfg, Xt_tr, Xe_tr, Xtau_tr, M_tr, y_tr, Xt_ev, Xe_ev, Xtau_ev, M_ev, y_ev,
               epochs=ABL_EPOCHS, measure_latency=False):
    c = {**DEF, **cfg}
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = DriftCapsNet(TEXT_DIM, EMO_DIM, prim_caps=c["prim_caps"], prim_dim=c["prim_dim"],
                         class_dim=c["class_dim"], d=c["fusion_dim"], gate=c["gate"], fusion=c["fusion"],
                         routing=c["routing"], dropout=c["dropout"]).to(DEVICE)
    w = class_weights(y_tr); focal = FocalLoss(w, gamma=c["focal_gamma"])
    ce = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=c["lr"], weight_decay=c["weight_decay"])
    dl = make_loaders(Xt_tr, Xe_tr, Xtau_tr, M_tr, y_tr)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    t0 = time.time(); model.train()
    for ep in range(epochs):
        for xt, xe, xtau, m, yb in dl:
            xt, xe, xtau, m, yb = [t.to(DEVICE) for t in (xt, xe, xtau, m, yb)]
            opt.zero_grad()
            if c["loss"] == "ce":
                loss = ce(model(xt, xe, xtau, m), yb)
            elif c["loss"] == "focal":
                loss = focal(model(xt, xe, xtau, m), yb)
            else:  # focal_contrastive
                logits, z = model(xt, xe, xtau, m, return_z=True)
                loss = focal(logits, yb) + c["contrast_w"] * supervised_contrastive(z, yb)
            loss.backward(); opt.step()
    train_secs = time.time() - t0

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(Xt_ev).to(DEVICE), torch.tensor(Xe_ev).to(DEVICE),
                       torch.tensor(Xtau_ev).to(DEVICE), torch.tensor(M_ev).to(DEVICE))
        prob = torch.softmax(logits, -1).cpu().numpy(); pred = logits.argmax(-1).cpu().numpy()
    try:
        auc = float(roc_auc_score(y_ev, prob, multi_class="ovr", average="macro", labels=[0,1,2,3]))
    except Exception:
        auc = None
    lat_ms = None
    if measure_latency:
        with torch.no_grad():
            tl = time.time()
            for _ in range(3):
                model(torch.tensor(Xt_ev).to(DEVICE), torch.tensor(Xe_ev).to(DEVICE),
                      torch.tensor(Xtau_ev).to(DEVICE), torch.tensor(M_ev).to(DEVICE))
            lat_ms = (time.time() - tl) / (3 * len(y_ev)) * 1000.0
    return {
        "accuracy": round(float(accuracy_score(y_ev, pred)), 4),
        "macro_f1": round(float(f1_score(y_ev, pred, average="macro", zero_division=0)), 4),
        "mcc": round(float(matthews_corrcoef(y_ev, pred)), 4),
        "tier3_recall": round(float(recall_score(y_ev, pred, labels=[3], average="macro", zero_division=0)), 4),
        "roc_auc": round(auc, 4) if auc is not None else None,
        "n_params": int(n_params), "train_secs": round(train_secs, 1),
        "epoch_ms": round(train_secs / epochs * 1000, 1),
        "infer_ms_per_sample": round(lat_ms, 4) if lat_ms is not None else None,
    }

log.info("train/eval driver ready (ABL_EPOCHS=%d)", ABL_EPOCHS)

2026-08-08 07:54:15,855 | INFO | train/eval driver ready (ABL_EPOCHS=40)


## 4. Ablations — vary one axis from the design-default baseline

All TRAIN, evaluated on the frozen TEST. These are **descriptive component comparisons at the a-priori
design defaults** — no model is *selected* on TEST. Reported on Tier-3 recall + macro-F1 + MCC.

In [5]:
ABL_AXES = {
    "routing": ["static", "dynamic"],
    "gate":    ["elu", "none", "sigmoid"],
    "fusion":  ["attention", "concat", "text_only"],
    "loss":    ["focal_contrastive", "focal", "ce"],
}
TR = (MURIL_TR, EMO_TR, TEMP_TR, MASK_TR, y_train)
TE = (MURIL_TE, EMO_TE, TEMP_TE, MASK_TE, y_test)

baseline = train_eval({}, *TR, *TE, measure_latency=True)
log.info("baseline: macroF1=%.3f mcc=%.3f T3rec=%.3f auc=%s",
         baseline["macro_f1"], baseline["mcc"], baseline["tier3_recall"], baseline["roc_auc"])

ablation = {"baseline": {"config": {k: DEF[k] for k in ("routing","gate","fusion","loss")}, **baseline}}
for axis, options in ABL_AXES.items():
    rows = []
    for opt_val in options:
        if opt_val == DEF[axis]:
            m = dict(baseline); tag = opt_val + " (baseline)"
        else:
            m = train_eval({axis: opt_val}, *TR, *TE)
            tag = opt_val
        rows.append({"value": tag, "macro_f1": m["macro_f1"], "mcc": m["mcc"],
                     "tier3_recall": m["tier3_recall"], "roc_auc": m["roc_auc"],
                     "d_macro_f1_vs_baseline": round(m["macro_f1"] - baseline["macro_f1"], 4)})
        log.info("  [%s=%s] macroF1=%.3f (Δ%+.3f) mcc=%.3f T3rec=%.3f",
                 axis, opt_val, m["macro_f1"], m["macro_f1"] - baseline["macro_f1"], m["mcc"], m["tier3_recall"])
    ablation[axis] = rows

print("=== ABLATIONS (frozen TEST; Δ vs baseline macro-F1) ===")
for axis in ABL_AXES:
    print(f"-- {axis} --")
    for r in ablation[axis]:
        print(f"   {r['value']:22} macroF1={r['macro_f1']:.3f} (Δ{r['d_macro_f1_vs_baseline']:+.3f}) "
              f"mcc={r['mcc']:+.3f} T3rec={r['tier3_recall']:.3f}")

2026-08-08 07:56:33,415 | INFO | baseline: macroF1=0.439 mcc=0.414 T3rec=0.043 auc=0.8197


2026-08-08 07:56:33,420 | INFO |   [routing=static] macroF1=0.439 (Δ+0.000) mcc=0.414 T3rec=0.043


2026-08-08 07:59:13,161 | INFO |   [routing=dynamic] macroF1=0.478 (Δ+0.039) mcc=0.463 T3rec=0.022


2026-08-08 07:59:13,165 | INFO |   [gate=elu] macroF1=0.439 (Δ+0.000) mcc=0.414 T3rec=0.043


2026-08-08 08:01:09,377 | INFO |   [gate=none] macroF1=0.540 (Δ+0.101) mcc=0.517 T3rec=0.109


2026-08-08 08:03:09,172 | INFO |   [gate=sigmoid] macroF1=0.383 (Δ-0.056) mcc=0.349 T3rec=0.022


... (3 outputs left out)

2026-08-08 08:06:56,034 | INFO |   [loss=focal_contrastive] macroF1=0.439 (Δ+0.000) mcc=0.414 T3rec=0.043


2026-08-08 08:09:10,526 | INFO |   [loss=focal] macroF1=0.522 (Δ+0.083) mcc=0.505 T3rec=0.043


2026-08-08 08:11:24,874 | INFO |   [loss=ce] macroF1=0.533 (Δ+0.094) mcc=0.517 T3rec=0.109


=== ABLATIONS (frozen TEST; Δ vs baseline macro-F1) ===
-- routing --
   static (baseline)      macroF1=0.439 (Δ+0.000) mcc=+0.414 T3rec=0.043
   dynamic                macroF1=0.478 (Δ+0.039) mcc=+0.463 T3rec=0.022
-- gate --
   elu (baseline)         macroF1=0.439 (Δ+0.000) mcc=+0.414 T3rec=0.043
   none                   macroF1=0.540 (Δ+0.101) mcc=+0.517 T3rec=0.109
   sigmoid                macroF1=0.383 (Δ-0.056) mcc=+0.349 T3rec=0.022
-- fusion --
   attention (baseline)   macroF1=0.439 (Δ+0.000) mcc=+0.414 T3rec=0.043
   concat                 macroF1=0.506 (Δ+0.067) mcc=+0.502 T3rec=0.043
   text_only              macroF1=0.351 (Δ-0.088) mcc=+0.180 T3rec=0.087
-- loss --
   focal_contrastive (baseline) macroF1=0.439 (Δ+0.000) mcc=+0.414 T3rec=0.043
   focal                  macroF1=0.522 (Δ+0.083) mcc=+0.505 T3rec=0.043
   ce                     macroF1=0.533 (Δ+0.094) mcc=+0.517 T3rec=0.109


## 5. Hyperparameter search — Optuna TPE or seeded random; tuned on TRAIN-internal val (no TEST leak)

The frozen TEST is **never** used for selection. We carve a **stratified 80/20 validation split from TRAIN**,
search to maximise **val macro-F1**, then retrain the best config on full TRAIN and score it **once** on TEST.

In [6]:
from sklearn.model_selection import train_test_split

tr_idx, va_idx = train_test_split(np.arange(len(y_train)), test_size=0.2, random_state=SEED, stratify=y_train)
def sub(arr, idx): return arr[idx]
TRsub = (sub(MURIL_TR,tr_idx), sub(EMO_TR,tr_idx), sub(TEMP_TR,tr_idx), sub(MASK_TR,tr_idx), y_train[tr_idx])
VAsub = (sub(MURIL_TR,va_idx), sub(EMO_TR,va_idx), sub(TEMP_TR,va_idx), sub(MASK_TR,va_idx), y_train[va_idx])
log.info("search split: train=%d val=%d (val tiers=%s)", len(tr_idx), len(va_idx),
         {int(t): int((y_train[va_idx]==t).sum()) for t in range(4)})

SPACE = {
    "prim_caps":  [16, 32], "prim_dim": [8, 16], "fusion_dim": [64, 128],
    "routing":    ["static", "dynamic"], "gate": ["none", "sigmoid", "elu"],
    "contrast_w": [0.0, 0.1, 0.2, 0.5], "lr": [5e-4, 1e-3, 2e-3], "dropout": [0.1, 0.3, 0.5],
}
def objective_cfg(sampler):
    return {k: sampler(k, v) for k, v in SPACE.items()}

trials = []
def run_trial(cfg):
    m = train_eval(cfg, *TRsub, *VAsub, epochs=OPT_EPOCHS)
    trials.append({"config": cfg, "val_macro_f1": m["macro_f1"], "val_mcc": m["mcc"]})
    return m["macro_f1"]

if HAS_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def obj(trial):
        cfg = objective_cfg(lambda k, v: trial.suggest_categorical(k, v))
        return run_trial(cfg)
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(obj, n_trials=N_TRIALS)
    best_cfg = study.best_params; best_val = study.best_value; SEARCH = "optuna_tpe"
else:
    rng = random.Random(SEED); SEARCH = "random_search (optuna unavailable; install optuna for TPE)"
    for _ in range(N_TRIALS):
        cfg = objective_cfg(lambda k, v: rng.choice(v))
        run_trial(cfg)
    best = max(trials, key=lambda t: t["val_macro_f1"]); best_cfg = best["config"]; best_val = best["val_macro_f1"]

log.info("search done (%s): best val macroF1=%.3f cfg=%s", SEARCH, best_val, best_cfg)
# retrain best on FULL train, score ONCE on TEST
best_test = train_eval(best_cfg, *TR, *TE, measure_latency=True)
log.info("BEST on frozen TEST: macroF1=%.3f mcc=%.3f auc=%s T3rec=%.3f",
         best_test["macro_f1"], best_test["mcc"], best_test["roc_auc"], best_test["tier3_recall"])
print("=== SEARCH (%s, %d trials; selection on TRAIN-val only) ===" % (SEARCH, len(trials)))
print("best val macro-F1:", round(best_val,4), "| best cfg:", best_cfg)
print("best config on frozen TEST -> macroF1=%.3f mcc=%.3f auc=%s T3rec=%.3f vs baseline macroF1=%.3f" % (
    best_test["macro_f1"], best_test["mcc"], best_test["roc_auc"], best_test["tier3_recall"], baseline["macro_f1"]))

2026-08-08 08:11:25,117 | INFO | search split: train=1576 val=395 (val tiers={0: 310, 1: 16, 2: 51, 3: 18})


2026-08-08 08:27:33,172 | INFO | search done (random_search (optuna unavailable; install optuna for TPE)): best val macroF1=0.912 cfg={'prim_caps': 16, 'prim_dim': 8, 'fusion_dim': 128, 'routing': 'dynamic', 'gate': 'elu', 'contrast_w': 0.5, 'lr': 0.001, 'dropout': 0.1}


2026-08-08 08:29:30,631 | INFO | BEST on frozen TEST: macroF1=0.488 mcc=0.475 auc=0.8516 T3rec=0.065


=== SEARCH (random_search (optuna unavailable; install optuna for TPE), 15 trials; selection on TRAIN-val only) ===
best val macro-F1: 0.9122 | best cfg: {'prim_caps': 16, 'prim_dim': 8, 'fusion_dim': 128, 'routing': 'dynamic', 'gate': 'elu', 'contrast_w': 0.5, 'lr': 0.001, 'dropout': 0.1}
best config on frozen TEST -> macroF1=0.488 mcc=0.475 auc=0.8516 T3rec=0.065 vs baseline macroF1=0.439


## 6. Efficiency + concept-drift / periodic-retraining policy

In [7]:
import os as _os
model_pt = OUTPUT_DIR / "model_state.pt"
efficiency = {
    "baseline_config": {k: DEF[k] for k in ("routing","gate","fusion","loss","prim_caps","prim_dim","fusion_dim")},
    "trainable_params_head": baseline["n_params"],
    "note_backbones": "MuRIL + " + EMO_MODEL_ID + " are FROZEN; their params are not trained and are "
                      "embedded once (cached). Trainable count is the fusion head only.",
    "train_seconds_total": baseline["train_secs"], "train_ms_per_epoch": baseline["epoch_ms"],
    "epochs": ABL_EPOCHS, "inference_ms_per_sample": baseline["infer_ms_per_sample"],
    "model_state_pt_bytes": int(_os.path.getsize(model_pt)) if model_pt.exists() else None,
    "device": DEVICE,
}

concept_drift_policy = {
    "principle": " — concept drift is monitored, not assumed static; periodic sliding-window retraining.",
    "monitor": "Track the notebook 08 cosine Drift Velocity on a rolling window of incoming posts per cohort; "
               "also monitor the predicted-tier distribution (population stability index) vs the training prior.",
    "trigger": "Retrain when (a) mean Drift Velocity over the window exceeds a control-chart threshold "
               "(e.g. > train_mean + 3σ), OR (b) PSI(tier distribution) > 0.2, OR (c) a fixed cadence elapses.",
    "cadence": "Default monthly sliding-window refresh; event-driven retrain on trigger (a)/(b).",
    "window": "Sliding window of the most recent W weeks (W tuned at scale-up); old data down-weighted, not dropped.",
    "what_retrains": "At pilot scale: the fusion head over frozen backbones (cheap). At scale-up: optionally "
                     "continued-MLM domain adaptation of MuRIL before head retrain.",
    "safeguards": "Hold out a frozen evaluation set per refresh; block deployment of a refreshed model that "
                  "regresses macro-F1/MCC; log every refresh (data window, metrics, drift trigger) for audit.",
    "status": "POLICY (written, ) — the deployment retraining loop is future work, not a smoke-stage build.",
}
print("efficiency:", json.dumps(efficiency, indent=1))
print("concept-drift policy keys:", list(concept_drift_policy))

efficiency: {
 "baseline_config": {
  "routing": "static",
  "gate": "elu",
  "fusion": "attention",
  "loss": "focal_contrastive",
  "prim_caps": 32,
  "prim_dim": 16,
  "fusion_dim": 128
 },
 "trainable_params_head": 805572,
 "note_backbones": "MuRIL + MilaNLProc/xlm-emo-t are FROZEN; their params are not trained and are embedded once (cached). Trainable count is the fusion head only.",
 "train_seconds_total": 133.5,
 "train_ms_per_epoch": 3338.7,
 "epochs": 40,
 "inference_ms_per_sample": 0.1155,
 "model_state_pt_bytes": 3228749,
 "device": "cpu"
}
concept-drift policy keys: ['principle', 'monitor', 'trigger', 'cadence', 'window', 'what_retrains', 'safeguards', 'status']


## 7. Persist reports

In [8]:
(OUTPUT_DIR / "ablation_report.json").write_text(json.dumps({
    "part": "stage3_part3.3_ablations", "version": PART_VERSION,
    "principle": "P1; P3 (no selection on TEST); P6 (component + efficiency justification); /3.3/3.4/3.5",
    "epochs": ABL_EPOCHS, "baseline_config": {k: DEF[k] for k in DEF},
    "ablations": ablation,
    "honest_note": "Pilot-scale (202 TRAIN; TEST Crisis 13): ablation deltas are INDICATIVE, not significant. "
                   "Descriptive comparisons at the a-priori design defaults (no model selected on TEST). The "
                   "full battery + significance = data scale-up + >=5 seeds (Objective 5).",
}, indent=2, ensure_ascii=False), encoding="utf-8")

(OUTPUT_DIR / "optuna_study.json").write_text(json.dumps({
    "part": "stage3_part3.3_search", "version": PART_VERSION, "method": SEARCH, "n_trials": len(trials),
    "selection_data": "TRAIN-internal stratified 80/20 val (frozen TEST NEVER used for selection, P3)",
    "search_space": SPACE, "trials": trials, "best_config": best_cfg, "best_val_macro_f1": round(float(best_val),4),
    "best_on_frozen_test": best_test, "baseline_on_frozen_test": baseline,
    "honest_note": "Small pilot-scale search (%d trials); the full >=100-trial Optuna TPE with pruning + >=5 "
                   "seeds is the documented scale-up. %s." % (
                   len(trials), "Optuna TPE used" if HAS_OPTUNA else "Optuna not installed -> seeded random search"),
}, indent=2, ensure_ascii=False), encoding="utf-8")

(OUTPUT_DIR / "efficiency.json").write_text(json.dumps(efficiency, indent=2), encoding="utf-8")
(OUTPUT_DIR / "concept_drift_policy.json").write_text(json.dumps(concept_drift_policy, indent=2), encoding="utf-8")
log.info("wrote ablation_report.json, optuna_study.json, efficiency.json, concept_drift_policy.json")

2026-08-08 08:29:30,684 | INFO | wrote ablation_report.json, optuna_study.json, efficiency.json, concept_drift_policy.json


## 8. Card & This step is complete when

**notebook 10 is complete when:** the **ablation_report** (routing/gate/fusion/loss, each on Tier-3 recall +
macro-F1 + MCC vs baseline), the **search** (Optuna TPE or seeded random, selected on TRAIN-val only — TEST
never used for selection), the **efficiency** line (params/time/latency/size), and the written **concept-drift
policy** all exist and are reproduced from a real run. Toggles independently switchable; honest
pilot-scale caveat. **No human; P3 leakage discipline (no selection on TEST).**

In [9]:
print("=" * 64); print("Objective 3 · notebook 10 — ABLATIONS / SEARCH / EFFICIENCY CARD"); print("=" * 64)
print("baseline (frozen TEST): macroF1=%.3f mcc=%.3f T3rec=%.3f auc=%s" % (
    baseline["macro_f1"], baseline["mcc"], baseline["tier3_recall"], baseline["roc_auc"]))
print("ablation axes         :", list(ABL_AXES))
print("search                : %s, %d trials | best val macroF1=%.3f -> TEST macroF1=%.3f mcc=%.3f" % (
    SEARCH.split(" ")[0], len(trials), best_val, best_test["macro_f1"], best_test["mcc"]))
print("efficiency            : %d head params, %.0f ms/epoch, %.4f ms/sample infer" % (
    efficiency["trainable_params_head"], efficiency["train_ms_per_epoch"], efficiency["inference_ms_per_sample"] or 0))
print("concept-drift policy  : written")
dod = all([(OUTPUT_DIR / f).exists() for f in
           ("ablation_report.json", "optuna_study.json", "efficiency.json", "concept_drift_policy.json")])
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE", "(pilot-scale; deltas indicative)")

Objective 3 · notebook 10 — ABLATIONS / SEARCH / EFFICIENCY CARD
baseline (frozen TEST): macroF1=0.439 mcc=0.414 T3rec=0.043 auc=0.8197
ablation axes         : ['routing', 'gate', 'fusion', 'loss']
search                : random_search, 15 trials | best val macroF1=0.912 -> TEST macroF1=0.488 mcc=0.475
efficiency            : 805572 head params, 3339 ms/epoch, 0.1155 ms/sample infer
concept-drift policy  : written

Completion check: PASS (pilot-scale; deltas indicative)
